In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

log_path = "rollout_log.csv"
df = pd.read_csv(log_path)

# Parse stringified arrays we still care about
for col in ["observation", "direct_joint_command"]:
    df[col] = df[col].apply(ast.literal_eval)

# Optional: inspect a single environment
# df = df[df["env_index"] == 0].reset_index(drop=True)

qpos = np.stack(df["observation"].apply(lambda obs: obs[:8]).to_numpy())
direct = np.stack(df["direct_joint_command"].apply(lambda cmd: cmd[:8]).to_numpy())
delta = direct - qpos
steps = df["step"].to_numpy()

plt.style.use("seaborn-v0_8-darkgrid")

series_style = {
    "qpos": {"color": "#1f77b4", "linewidth": 1.4, "alpha": 1.0, "label": "qpos (blue)"},
    "direct": {"color": "#2ca02c", "linewidth": 1.2, "alpha": 0.85, "label": "direct joint command (green)"},
    "delta": {"color": "#ff7f0e", "linewidth": 1.2, "alpha": 0.85, "label": "delta (orange)"},
}

fig, axes = plt.subplots(4, 2, figsize=(18, 12), sharex=True)
joint_labels = [f"Joint {idx}" for idx in range(8)]

for idx, ax in enumerate(axes.flat):
    ax.plot(steps, qpos[:, idx], **series_style["qpos"])
    ax.plot(steps, direct[:, idx], **series_style["direct"])
    ax.set_title(joint_labels[idx])
    ax.set_ylabel("Value")

axes[-1, 0].set_xlabel("Step")
axes[-1, 1].set_xlabel("Step")
handles = [plt.Line2D([], [], **series_style["qpos"]),
           plt.Line2D([], [], **series_style["direct"])]
fig.legend(handles, [h.get_label() for h in handles], loc="upper right")
fig.suptitle("First Eight qpos vs Direct Joint Commands", fontsize=18, y=1.02)
fig.tight_layout()
plt.show()

# Plot deltas separately
fig, axes = plt.subplots(4, 2, figsize=(18, 12), sharex=True)
for idx, ax in enumerate(axes.flat):
    ax.plot(steps, delta[:, idx], **series_style["delta"])
    ax.axhline(0.0, color="grey", linewidth=0.8)
    ax.set_title(f"Joint {idx} delta")
    ax.set_ylabel("qpos delta")
axes[-1, 0].set_xlabel("Step")
axes[-1, 1].set_xlabel("Step")
fig.suptitle("Direct Command Delta (command - qpos)", fontsize=18, y=1.02)
fig.tight_layout()
plt.show()

corr = pd.DataFrame({
    "joint": [f"joint_{idx}" for idx in range(8)],
    "pearson_corr_direct": [np.corrcoef(qpos[:, idx], direct[:, idx])[0, 1] for idx in range(8)],
    "pearson_corr_delta": [np.corrcoef(qpos[:, idx], delta[:, idx])[0, 1] for idx in range(8)],
})
display(corr)
